# LPCMCI motorneuron baseline

Run this notebook independently of the motorneuron `oasis.ipynb`. Install the optional PAG environment first with `uv sync --extra pag`. Results are descriptive because the recordings have no directed ground truth; raw PAG tensors remain primary.

<!-- reviewer-resume-contract -->
## Execution and resume contract

Each fluorescence-type–recording–representation fit is checkpointed. Inputs are the exact deduplicated records read by the c-GC/c-GC* motorneuron notebooks. Re-run the identical cell after interruption; configuration and input-file digests must match.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'empirical_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))

RUN_LPCMCI = True
FLUO_TYPES = 'dff,f_smooth'
RECORDINGS = 'F3T1,F3T2,F5T2'
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/revision_campaign/motorneurons_lpcmci'
command = [
    RUNNER_PYTHON, 'examples/empirical_baselines.py',
    '--components', 'lpcmci', '--fluo-types', FLUO_TYPES,
    '--recordings', RECORDINGS,
    '--representations', 'full,deconvolved,rise,fall,fall_residual',
    '--output-dir', str(OUTPUT_DIR), '--resume',
]
print(shlex.join(command))
if RUN_LPCMCI:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))